# Step 17 — The remaining revisions, end to end (no dependency on earlier runs)

Everything the second review asked for on the MNIST side, in one notebook. Parts A–D and F are CPU; part E trains two small denoisers (≈2 min each on a T4) and needs the GPU.

- **A. Bootstrap intervals.** 1,000 replicates for $h$ at each pair's transition (resampling both classes), plus 100 replicates with re-bisection to show the widths agree.
- **B. Matched floor.** 180 symmetrised controls per dataset (4 per pair), each using the real pair's direction and scale with class $b$ replaced by the mirror image of an independent half of class $a$; the 95th percentile of $|h|$ is the floor. Sample size per control is stated.
- **C. Exponent calibration.** The identical pipeline on synthetic $d=784$ mixtures with known width and weight asymmetry: does a finite fit window reproduce a slope near $0.36$?
- **D. Full-dimensional samplers.** A $d=784$ mixture with unequal class widths and analytic score; real Heun and SDE samplers in $d=784$; class-fraction error vs $|h|$ compared with the one-dimensional prediction.
- **E. Intervention controls** (GPU): sham replacement, random-direction replacement, classifier confidence, on (0,1) and (3,8) at epoch 20.
- **F. The VP-offset outlier**: which pair, and its variance curve.

Internet on (MNIST download). ~1 h.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time, copy
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
from scipy.stats import theilslopes, spearmanr
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); torch.manual_seed(0); plt.rcParams['figure.dpi'] = 120
def load(ds_cls, root):
    d = ds_cls(root, train=True, download=True); return (d.data.numpy().astype(np.float32)/127.5 - 1.0).reshape(-1, 784), d.targets.numpy()
X_MN, Y_MN = load(datasets.MNIST, '/kaggle/working/mnist'); X_FM, Y_FM = load(datasets.FashionMNIST, '/kaggle/working/fmnist')

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau, grid=GRID):
    dg = grid[1]-grid[0]; h, _ = np.histogram(m0, bins=np.append(grid-dg/2, grid[-1]+dg/2), density=True); k = np.exp(-grid**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau): p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d_ = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d_[:-1]) != np.sign(d_[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d_[i]*DG/(d_[i+1]-d_[i])) if d_[i+1] != d_[i] else float(GRID[i]), 'max' if d_[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.05, 3.0, 60), iters=12):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(Xa.shape[1]), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced(X, Y, pair):
    Xa = X[Y == pair[0]]; Xb = X[Y == pair[1]]; n = min(len(Xa), len(Xb)); return Xa[:n], Xb[:n]
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def analyse_proj(m0):
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    if bj is None: return None
    return dict(tau=bj[0], jump=bj[1], **cusp_fit(*exact_drift(m0, bj[0])))
def project(Xa, Xb):
    u, mid, half = lda_direction(Xa, Xb); return (Xa-mid) @ u / half, (Xb-mid) @ u / half

## A. Bootstrap intervals for h (1,000 replicates at the transition; 100 with re-bisection)

In [ ]:
def census_with_ci(X, Y, name, nboot_fixed=1000, nboot_rebisect=100):
    out = []; t0 = time.time()
    for pair in combinations(range(10), 2):
        Xa, Xb = balanced(X, Y, pair); pa, pb = project(Xa, Xb); m0 = np.concatenate([pa, pb]); r = analyse_proj(m0)
        if r is None: continue
        hs = []
        for _ in range(nboot_fixed):                              # resample both classes; refit at the pair's tau_bif
            mb = np.concatenate([pa[rng.integers(len(pa), size=len(pa))], pb[rng.integers(len(pb), size=len(pb))]]); hs.append(cusp_fit(*exact_drift(mb, r['tau']))['h'])
        hs = np.array(hs); lo, hi = np.percentile(hs, [2.5, 97.5])
        hr = []
        for _ in range(nboot_rebisect):                           # resample and re-bisect
            mb = np.concatenate([pa[rng.integers(len(pa), size=len(pa))], pb[rng.integers(len(pb), size=len(pb))]]); rb = analyse_proj(mb)
            if rb: hr.append(rb['h'])
        lo2, hi2 = np.percentile(hr, [2.5, 97.5]) if len(hr) > 20 else (np.nan, np.nan)
        out.append(dict(pair=pair, **{k: r[k] for k in ['tau', 'jump', 'h', 'a', 'b', 'clamped']}, h_lo=lo, h_hi=hi, h_lo2=lo2, h_hi2=hi2, sd=hs.std()))
    print(f"{name}: {len(out)} pairs, {time.time()-t0:.0f}s; median CI half-width (fixed tau) {np.median([(o['h_hi']-o['h_lo'])/2 for o in out]):.4f}, (re-bisect) {np.nanmedian([(o['h_hi2']-o['h_lo2'])/2 for o in out]):.4f}")
    return out
MN = census_with_ci(X_MN, Y_MN, 'MNIST'); FM = census_with_ci(X_FM, Y_FM, 'FashionMNIST')

## B. Matched-size floor from many symmetrised controls

In [ ]:
def floor_controls(X, Y, name, per_pair=4):
    """Symmetrised controls with a REAL pair's direction and scale: class b replaced by the mirror image of an independent half of class a.
    Symmetric in distribution, real sampling noise, same direction/normalisation as the real pair. Sample size per 'class' is half a class."""
    hs = []; sizes = []
    for pair in combinations(range(10), 2):
        Xa, Xb = balanced(X, Y, pair); u, mid, half = lda_direction(Xa, Xb); pa = (Xa-mid) @ u / half
        for _ in range(per_pair):
            idx = rng.permutation(len(pa)); k = len(pa)//2; m0 = np.concatenate([pa[idx[:k]], -pa[idx[k:2*k]]]); sizes.append(k)
            r = analyse_proj(m0)
            if r: hs.append(abs(r['h']))
    hs = np.array(hs); print(f"{name}: {len(hs)} controls of {int(np.median(sizes))} samples per class (real pairs: ~{min(np.bincount(Y))}); |h| median {np.median(hs):.4f}, 95th pct {np.percentile(hs, 95):.4f}, max {hs.max():.4f}")
    return float(np.percentile(hs, 95))
FLOOR_MN = floor_controls(X_MN, Y_MN, 'MNIST'); FLOOR_FM = floor_controls(X_FM, Y_FM, 'FashionMNIST')
for name, cen, fl in [('MNIST', MN, FLOOR_MN), ('FashionMNIST', FM, FLOOR_FM)]:
    above = [o for o in cen if abs(o['h']) > fl and (o['h_lo'] > 0 or o['h_hi'] < 0)]
    print(f"{name}: pairs with |h| above the matched floor AND 95% CI excluding zero: {len(above)}/{len(cen)}; median |h| {np.median([abs(o['h']) for o in cen]):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, (name, cen, fl) in enumerate([('MNIST', MN, FLOOR_MN), ('FashionMNIST', FM, FLOOR_FM)]):
    o = sorted(cen, key=lambda r: abs(r['h'])); hv = np.array([r['h'] for r in o]); lo = np.array([r['h_lo'] for r in o]); hi = np.array([r['h_hi'] for r in o])
    ax[k].errorbar(range(len(o)), hv, yerr=[np.maximum(hv-lo, 0), np.maximum(hi-hv, 0)], fmt='o', ms=3); ax[k].axhspan(-fl, fl, color='k', alpha=0.12, label='matched floor'); ax[k].axhline(0, color='k', lw=0.5)
    ax[k].set_title(f'{name}: h with 1,000-replicate 95% CI'); ax[k].set_xlabel('pair (sorted by |h|)'); ax[k].legend(fontsize=7)
plt.tight_layout(); plt.show()

## C. Exponent calibration on synthetic mixtures with known asymmetry

In [ ]:
def synth_pair(n=6000, d=784, width_a=0.15, width_b=0.15, w=0.5, seed=0, iso=0.3):
    """Two classes at +/-s with isotropic pixel noise (iso) plus a class-specific spread ALONG the class direction (width_a, width_b, in units
    where the class means are at +/-1). Width asymmetry along the class axis is what produces h; isotropic width asymmetry projects to ~0 in d=784."""
    r = np.random.default_rng(seed); s = np.sign(r.standard_normal(d)).astype(np.float32); na = int(2*n*w); nb_ = 2*n - na
    A = s[None, :]*(1 + width_a*r.standard_normal((na, 1))).astype(np.float32) + iso*r.standard_normal((na, d)).astype(np.float32)
    B = -s[None, :]*(1 + width_b*r.standard_normal((nb_, 1))).astype(np.float32) + iso*r.standard_normal((nb_, d)).astype(np.float32); return A, B
rows = []
for wa, wb in [(0.15, 0.15), (0.15, 0.17), (0.15, 0.2), (0.15, 0.24), (0.15, 0.3), (0.15, 0.36), (0.15, 0.45), (0.15, 0.55), (0.15, 0.7), (0.15, 0.9)]:
    for seed in [0, 1]:
        A, B = synth_pair(width_a=wa, width_b=wb, seed=seed); pa, pb = project(A, B); r = analyse_proj(np.concatenate([pa, pb]))
        if r: rows.append(dict(kind='width', ratio=wb/wa, **r))
for w in [0.5, 0.52, 0.55, 0.6, 0.65, 0.7, 0.75]:
    for seed in [0, 1]:
        A, B = synth_pair(w=w, seed=seed); pa, pb = project(A, B); r = analyse_proj(np.concatenate([pa, pb]))
        if r: rows.append(dict(kind='weight', ratio=w, **r))
for kind in ['width', 'weight', 'both']:
    S = [r for r in rows if (kind == 'both' or r['kind'] == kind) and abs(r['h']) > FLOOR_MN and not r['clamped']]
    if len(S) < 5: print(f"{kind}: too few points ({len(S)})"); continue
    x = np.log([abs(r['h'])/(2*r['b']) for r in S]); y = np.log([r['jump'] for r in S]); sl, ic = np.polyfit(x, y, 1); ts = theilslopes(y, x)
    print(f"synthetic {kind:>6} asymmetry: n={len(S)}, |h| range {min(abs(r['h']) for r in S):.3f}-{max(abs(r['h']) for r in S):.3f}; OLS slope {sl:.3f}, Theil-Sen {ts[0]:.3f} [{ts[2]:.3f}, {ts[3]:.3f}]; median jump/3(|h|/2b)^(1/3) = {np.median([r['jump']/(3*(abs(r['h'])/(2*r['b']))**(1/3)) for r in S]):.2f}")
print("Interpretation: if the synthetic slopes are also ~0.36 with ratio ~0.9, the MNIST exponent and the 10% shortfall are finite-window effects of the cubic fit, not properties of the data.")
S = [r for r in MN if abs(r['h']) > FLOOR_MN and not r['clamped']]; x = np.log([abs(r['h'])/(2*r['b']) for r in S]); y = np.log([r['jump'] for r in S]); sl, ic = np.polyfit(x, y, 1)
print(f"MNIST (this run): slope {sl:.3f}; with slope fixed at 1/3 the best prefactor is {np.exp(np.mean(y - x/3)):.2f} (theory 3)")

## D. Full-dimensional samplers on a mixture with analytic score (d = 784)

In [ ]:
def make_mix(d=784, wa=0.15, wb=0.15, iso=0.3, seed=0):
    """Mixture 1/2 N(+s, S_a) + 1/2 N(-s, S_b), S_c = iso^2 I + (w_c^2 d) uu^T with u = s/|s|: same family as synth_pair. Analytic noised score."""
    r = np.random.default_rng(seed); s = np.sign(r.standard_normal(d)).astype(np.float64); uh = s/np.sqrt(d)
    def score(x, sigma):
        a = iso**2 + sigma**2; out_w = []; out_g = []
        for mu, wc in [(s, wa), (-s, wb)]:
            b = wc**2*d; dx = x - mu; proj = dx @ uh
            quad = ((dx**2).sum(1) - (b/(a+b))*proj**2)/a; logdet = (d-1)*np.log(a) + np.log(a+b)
            out_w.append(-0.5*quad - 0.5*logdet); out_g.append(-(dx - (b/(a+b))*proj[:, None]*uh[None, :])/a)
        m = np.maximum(out_w[0], out_w[1]); pa = np.exp(out_w[0]-m); pb = np.exp(out_w[1]-m); wgt = (pa/(pa+pb))[:, None]
        return wgt*out_g[0] + (1-wgt)*out_g[1]
    return s, score
def sched(steps, s_max=20.0, s_min=0.002, rho=7.0):
    i = np.arange(steps); return np.append((s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho)-s_max**(1/rho)))**rho, 0.0)
def sample_full(score, d, steps, kind='heun', beta=1.0, n=4000, seed=1):
    r = np.random.default_rng(seed); sig = sched(steps); x = r.standard_normal((n, d))*sig[0]
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]
        if kind == 'heun':
            d0 = -s0*score(x, s0); x1 = x + (s1-s0)*d0
            if s1 > 0: d1 = -s1*score(x1, s1); x = x + (s1-s0)*0.5*(d0+d1)
            else: x = x1
        else:
            x = x + (s1-s0)*(-(1+beta**2)*s0*score(x, s0)) + beta*np.sqrt(2*s0*abs(s1-s0))*r.standard_normal((n, d))
    return x
print(f"{'width ratio':>12} {'h (LDA census)':>15} {'exact frac':>11} {'heun10':>8} {'heun40':>8} {'sde10':>8} {'sde40':>8} {'sde160':>8}   (class-fraction error toward the persisting branch)")
D = []
for wb in [0.15, 0.2, 0.3, 0.45, 0.7]:
    s, score = make_mix(wb=wb); A, B = synth_pair(width_a=0.15, width_b=wb, seed=0); pa, pb = project(A, B); r = analyse_proj(np.concatenate([pa, pb])); h = r['h'] if r else np.nan
    # exact class fraction of the mixture (balanced by construction) along s: fraction with s.x > 0 on clean data
    frac_exact = 0.5*(( (A @ s) > 0).mean() + ((B @ s) > 0).mean())
    row = dict(ratio=wb/0.15, h=h, exact=frac_exact)
    for name, kw in [('heun10', dict(steps=10)), ('heun40', dict(steps=40)), ('sde10', dict(steps=10, kind='sde')), ('sde40', dict(steps=40, kind='sde')), ('sde160', dict(steps=160, kind='sde'))]:
        x = sample_full(score, 784, **kw); fr = ((x @ s) > 0).mean(); row[name] = (fr - frac_exact)*np.sign(h) if np.isfinite(h) else np.nan
    D.append(row); print(f"{wb/0.15:12.2f} {h:15.4f} {frac_exact:11.3f} " + ' '.join(f"{row[k]:+8.4f}" for k in ['heun10', 'heun40', 'sde10', 'sde40', 'sde160']))
print("Compare with the reduced-dynamics prediction (Section 6): Heun ~0; SDE at 10 steps ~ -0.28 |h|; SDE at 40 ~ -0.03 |h|; SDE at 160 ~ 0. Sign convention: positive = toward the branch that persists.")

## E. Intervention controls (GPU): sham replacement, random direction, classifier confidence

In [ ]:
SIGMA_DATA = 0.5
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__(); self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1); self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1); self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31)); e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def train(pair, epochs=20, bs=128, lr=2e-4, P_mean=-0.6, P_std=1.4, ema_decay=0.999, seed=0):
    torch.manual_seed(seed); X = torch.from_numpy(X_MN[np.isin(Y_MN, pair)]).view(-1, 1, 28, 28).to(dev)
    net = UNet().to(dev); ema = copy.deepcopy(net); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(X); t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n-bs+1, bs):
            x0 = X[perm[i:i+bs]]; sigma = torch.exp(P_mean + P_std*torch.randn(bs, device=dev)); xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0)
            w = (sigma**2 + SIGMA_DATA**2)/(sigma*SIGMA_DATA)**2; loss = (w.view(-1,1,1,1)*(denoise(net, xt, sigma) - x0)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()): pe.lerp_(pn, 1-ema_decay)
    print(f"  trained {pair} ({time.time()-t0:.0f}s)"); return ema.eval()
class CNN(nn.Module):
    def __init__(self):
        super().__init__(); self.c1 = nn.Conv2d(1, 32, 3, padding=1); self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(64*7*7, 2)
    def forward(self, x): x = F.max_pool2d(F.relu(self.c1(x)), 2); x = F.max_pool2d(F.relu(self.c2(x)), 2); return self.fc(x.flatten(1))
def train_cnn(Xa, Xb, epochs=3):
    X = torch.from_numpy(np.concatenate([Xa, Xb])).view(-1, 1, 28, 28).to(dev); Y = torch.cat([torch.zeros(len(Xa)), torch.ones(len(Xb))]).long().to(dev); net = CNN().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), 256): idx = perm[i:i+256]; loss = F.cross_entropy(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    return net.eval()
S_MAX, S_MIN, RHO = 20.0, 0.002, 7.0; SIG_TAB = np.geomspace(0.05, S_MAX, 48)
@torch.no_grad()
def drift_tables(net, Xflat, u, mid, half, n=20000, nbins=80):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); m0 = (Xflat-mid) @ u / half; tabs = []
    for sigma in SIG_TAB:
        x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
        D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), float(sigma), device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
        m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
        lo, hi = np.quantile(m, [0.001, 0.999]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
        cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
        cen = 0.5*(edges[:-1]+edges[1:]); gnet = np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
        c = np.linspace(min(lo, -30), max(hi, 30), 12001); p = smooth_hist(m0, sigma**2/half**2, grid=c); gex = np.interp(cen, c, np.gradient(np.log(p), c)); tabs.append((cen, gex, gnet))
    return tabs
def make_corr(tabs, mode):
    """mode 'exact': g_exact - g_net ; 'sham': g_net' - g_net with an independent re-estimate (expected ~0) ; 'none': 0."""
    def corr(m_np, sigma):
        k = int(np.argmin(np.abs(np.log(SIG_TAB) - np.log(sigma)))); cen, gex, gnet = tabs[k]
        if mode == 'exact': return np.interp(m_np, cen, gex - gnet)
        if mode == 'sham': return np.interp(m_np, cen, tabs_sham[k][2] - gnet)
        return np.zeros_like(m_np)
    return corr
@torch.no_grad()
def sample(net, u, mid, half, corr=None, steps=40, churn=0.0, n=4000, direction=None):
    ut = torch.from_numpy(direction if direction is not None else u).to(dev); midt = torch.from_numpy(mid).to(dev); ut4 = ut.view(1, 1, 28, 28)
    um = torch.from_numpy(u).to(dev)   # the order parameter is always measured along u
    def Dc(x, s):
        D = denoise(net, x, s.expand(n))
        if corr is None: return D
        m = ((x.view(n, -1) - midt) @ um / half).cpu().numpy(); dg = torch.from_numpy(corr(m, float(s)).astype(np.float32)).to(dev); return D + (float(s)**2/half)*dg.view(n, 1, 1, 1)*ut4
    i = torch.arange(steps, device=dev); sig = torch.cat([(S_MAX**(1/RHO) + i/(steps-1)*(S_MIN**(1/RHO)-S_MAX**(1/RHO)))**RHO, torch.zeros(1, device=dev)]); x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]
        if churn > 0 and 0.01 <= s0 <= S_MAX: gam = min(churn/steps, 2**0.5-1); sh = s0*(1+gam); x = x + torch.sqrt(sh**2 - s0**2)*torch.randn_like(x); s0 = sh
        d0 = (x - Dc(x, s0))/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - Dc(x1, s1))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return x
print(f"{'pair':>6} {'sampler':>8} {'condition':>18} {'frac+ CNN':>10} {'mean conf':>10} {'mean entropy':>13}")
for pair in [(0, 1), (3, 8)]:
    Xa, Xb = balanced(X_MN, Y_MN, pair); u, mid, half = lda_direction(Xa, Xb); Xflat = np.concatenate([Xa, Xb]); cnn = train_cnn(Xa, Xb); net = train(pair)
    tabs = drift_tables(net, Xflat, u, mid, half); tabs_sham = drift_tables(net, Xflat, u, mid, half)      # independent re-estimate for the sham
    rnd = rng.standard_normal(784).astype(np.float32); rnd -= (rnd @ u)*u; rnd /= np.linalg.norm(rnd)  # random direction orthogonal to u
    for sname, kw in [('heun40', dict(steps=40)), ('sde40', dict(steps=40, churn=40)), ('sde80', dict(steps=80, churn=40))]:
        for cond, corr, direction in [('none', None, None), ('exact replacement', make_corr(tabs, 'exact'), None), ('sham replacement', make_corr(tabs, 'sham'), None), ('random direction', make_corr(tabs, 'exact'), rnd)]:
            x = sample(net, u, mid, half, corr, direction=direction, **kw); logits = cnn(x); pr = F.softmax(logits, 1)
            print(f"{pair!s:>6} {sname:>8} {cond:>18} {float((logits.argmax(1) == 0).float().mean()):10.3f} {float(pr.max(1).values.mean()):10.3f} {float(-(pr*torch.log(pr+1e-12)).sum(1).mean()):13.3f}")

## F. The VP-offset outlier

In [ ]:
def varH_curve(pa, pb, taus, n=60000):
    out = []
    for tau in taus:
        A_, B_ = smooth_hist(pa, tau), smooth_hist(pb, tau); m = np.concatenate([pa[rng.choice(len(pa), n//2)], pb[rng.choice(len(pb), n//2)]]) + np.sqrt(tau)*rng.standard_normal(n)
        ia = np.clip(((m-GRID[0])/DG).round().astype(int), 0, len(GRID)-1); pc = np.clip(A_[ia]/(A_[ia]+B_[ia]), 1e-12, 1-1e-12); out.append((-(pc*np.log(pc)+(1-pc)*np.log(1-pc))).var())
    return np.array(out)
taus = np.geomspace(0.05, 5.0, 60); res = []
for o in MN:
    Xa, Xb = balanced(X_MN, Y_MN, o['pair']); pa, pb = project(Xa, Xb); v = varH_curve(pa, pb, taus); tp = taus[int(np.argmax(v))]; res.append((o['pair'], o['tau'], tp, v))
ratios = np.array([r[2]/r[1] for r in res]); dts = 0.5*np.log(ratios)
print(f"tau_peak/tau_bif: median {np.median(ratios):.2f}, range {ratios.min():.2f}-{ratios.max():.2f}; VP offset dt median {np.median(dts):+.2f}, range {dts.min():+.2f} to {dts.max():+.2f}")
worst = res[int(np.argmax(ratios))]; print(f"largest ratio: pair {worst[0]} (tau_bif {worst[1]:.3f}, tau_peak {worst[2]:.3f}); flatness of its curve: max/median over tau in [0.3,3] = {worst[3][(taus>0.3)&(taus<3)].max()/np.median(worst[3][(taus>0.3)&(taus<3)]):.3f}")
plt.figure(figsize=(5, 3.4)); plt.plot(taus, worst[3], 'o-', ms=3, label=str(worst[0])); typical = res[int(np.argsort(ratios)[len(ratios)//2])]; plt.plot(taus, typical[3], 's-', ms=3, label=f'{typical[0]} (typical)')
plt.axvline(worst[1], color='C0', ls='--', lw=0.8); plt.axvline(typical[1], color='C1', ls='--', lw=0.8); plt.xscale('log'); plt.xlabel('tau'); plt.ylabel('Var[H(c|m)]'); plt.legend(); plt.tight_layout(); plt.show()

## What to report back
**A/B.** The two census lines (1,000-replicate CI widths; counts above the matched floor with CI excluding zero) and the floor lines with control sample sizes.
**C.** The synthetic exponent lines: whether width- and weight-asymmetry controls give a slope near 0.36 and a ratio near 0.9 with the same pipeline.
**D.** The full-dimensional table: does the stochastic-sampler bias toward the persisting branch scale with |h| in d=784 as the reduced dynamics predicted, and is the deterministic error zero?
**E.** The controls table: sham ≈ none, random direction ≈ none, exact replacement corrects; and whether confidence/entropy change under correction.
**F.** The outlier pair and its flatness ratio.